# AtlasTransformerV6: DepMap Pre-training + Patient Fine-tuning

Self-contained notebook for GPU training. No Neo4j required.

**Upload these files first** (from `gnn/results/colab_export/`):
- `colab_depmap_data.pt`
- `colab_patient_data.pt`
- `atlas_transformer_v6.py`
- `essentiality_head.py`
- `cox_sage.py`

In [ ]:
# Install deps
!pip install -q scikit-survival

In [ ]:
# Option A: Mount Google Drive (recommended for 840MB patient data)
# Upload files to Google Drive first: MyDrive/okg_colab/
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/okg_colab'

needed = ['colab_depmap_data.pt', 'colab_patient_data.pt',
          'atlas_transformer_v6.py', 'essentiality_head.py', 'cox_sage.py']

if os.path.exists(DRIVE_DIR):
    for f in needed:
        src = os.path.join(DRIVE_DIR, f)
        if os.path.exists(src) and not os.path.exists(f):
            print(f'Copying {f}...')
            shutil.copy2(src, f)
    print('Done copying from Drive')
else:
    print(f'{DRIVE_DIR} not found — upload files there first, or use Option B')

# Option B: Direct upload (slow for large files, uncomment if needed)
# from google.colab import files
# missing = [f for f in needed if not os.path.exists(f)]
# if missing:
#     print(f'Upload: {missing}')
#     uploaded = files.upload()

missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print(f'WARNING: still missing: {missing}')
else:
    print('All files present ✓')

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json, time
from collections import Counter
from sklearn.model_selection import StratifiedKFold, train_test_split
from sksurv.metrics import concordance_index_censored

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Import models
from atlas_transformer_v6 import AtlasTransformerV6
from essentiality_head import EssentialityHead, EssentialityLoss

def cox_ph_loss(hazard, time, event):
    """Cox partial likelihood loss (Breslow approximation)."""
    order = torch.argsort(time, descending=True)
    hazard = hazard[order]
    event = event[order].float()
    log_cumsum_h = torch.logcumsumexp(hazard, dim=0)
    loss = -torch.mean((hazard - log_cumsum_h) * event)
    return loss

def sign_loss(model, node_features, node_mask):
    """Supervise per-mutation sign prediction from atlas log_hr.

    feat[0] = log_hr: positive = harmful, negative = protective.
    The sign head should predict this direction from the learned representation.
    Only supervise mutations with non-zero log_hr (have atlas signal).
    """
    sign_logits = model._last_sign_logits  # (B, N)
    log_hr = node_features[:, :, 0]        # (B, N) — ground truth sign

    # Only supervise where we have atlas signal AND real nodes
    has_signal = (log_hr.abs() > 0.05) & (node_mask > 0)
    if has_signal.sum() < 10:
        return torch.tensor(0.0, device=sign_logits.device)

    # Target: +1 harmful, -1 protective
    target = torch.sign(log_hr)
    # Convert to [0, 1] for BCE: harmful=1, protective=0
    target_01 = (target[has_signal] + 1) / 2  # {-1,+1} → {0,1}
    pred = sign_logits[has_signal]

    return nn.functional.binary_cross_entropy_with_logits(pred, target_01)

## Shared utilities

In [ ]:
def gather_edge_features(gene_pair_matrix, patient_edge_feats, gene_indices, gene_masks):
    """Gather gene-pair + patient features into (B, N, N, D) tensor."""
    B, N = gene_indices.shape
    safe_idx = gene_indices.clamp(0, gene_pair_matrix.shape[0] - 1)
    idx_i = safe_idx.unsqueeze(2).expand(B, N, N)
    idx_j = safe_idx.unsqueeze(1).expand(B, N, N)
    graph_feats = gene_pair_matrix[idx_i, idx_j]
    pair_mask = (gene_masks.unsqueeze(1) * gene_masks.unsqueeze(2)).unsqueeze(-1)
    graph_feats = graph_feats * pair_mask
    return torch.cat([graph_feats, patient_edge_feats * pair_mask], dim=-1)

---
## Stage 1: DepMap Pre-training

In [ ]:
# Load DepMap data
dm = torch.load('colab_depmap_data.pt', map_location='cpu', weights_only=False)
print(f"Cell lines: {dm['node_features'].shape[0]}")
print(f"Valid essentiality targets: {int(dm['essentiality_masks'].sum())}")
print(f"Gene vocab: {len(dm['gene_vocab'])}")
print(f"Edge dim: {dm['graph_edge_dim']} + {dm['patient_edge_dim']} = {dm['graph_edge_dim'] + dm['patient_edge_dim']}")

In [ ]:
# Pre-training config
PRETRAIN_CFG = {
    'hidden_dim': 64,
    'n_heads': 4,
    'n_intra_layers': 2,
    'dropout': 0.2,
    'node_feat_dim': dm['node_features'].shape[-1],  # auto-detect from data
    'edge_feat_dim': dm['graph_edge_dim'] + dm['patient_edge_dim'],
    'n_cancer_types': dm['n_cancer_types'],
    'n_channels': dm['n_channels'],
    'n_blocks': dm['n_blocks'],
}

PRETRAIN_EPOCHS = 200
PRETRAIN_LR = 1e-3
PRETRAIN_BATCH = 32
PRETRAIN_PATIENCE = 25
RANK_WEIGHT = 0.3
MUT_DROP_MIN = 0.1
MUT_DROP_MAX = 0.3
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Node feat dim: {PRETRAIN_CFG["node_feat_dim"]}')

In [ ]:
def apply_mutation_dropout(nf, nm, ef, ess, em, drop_min=0.1, drop_max=0.3):
    """Randomly mask mutations during training."""
    B, N, _ = nf.shape
    nf, nm, ef, ess, em = nf.clone(), nm.clone(), ef.clone(), ess.clone(), em.clone()
    for b in range(B):
        real = nm[b].bool()
        n_real = real.sum().item()
        if n_real <= 1:
            continue
        drop_rate = np.random.uniform(drop_min, drop_max)
        n_drop = max(1, int(n_real * drop_rate))
        n_drop = min(n_drop, n_real - 1)
        real_indices = real.nonzero(as_tuple=True)[0]
        drop_indices = real_indices[torch.randperm(len(real_indices))[:n_drop]]
        nm[b, drop_indices] = 0.0
        nf[b, drop_indices] = 0.0
        ef[b, drop_indices, :] = 0.0
        ef[b, :, drop_indices] = 0.0
        em[b, drop_indices] = 0.0
    return nf, nm, ef, ess, em

In [ ]:
# Train/val split
from torch.utils.data import DataLoader, TensorDataset

N_cl = len(dm['cancer_types'])
indices = np.arange(N_cl)
ct_np = dm['cancer_types'].numpy()
ct_counts = Counter(ct_np)
strat_labels = np.array([ct if ct_counts[ct] >= 5 else -1 for ct in ct_np])
rare_count = (strat_labels == -1).sum()
use_stratify = strat_labels if rare_count >= 2 else None

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=SEED, stratify=use_stratify)
train_idx = torch.tensor(train_idx, dtype=torch.long)
val_idx = torch.tensor(val_idx, dtype=torch.long)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}')

# Move data to device
dm_gpm = dm['gene_pair_matrix'].to(device)

train_ds = TensorDataset(
    dm['node_features'][train_idx], dm['node_masks'][train_idx],
    dm['cancer_types'][train_idx], dm['clinical'][train_idx],
    dm['gene_indices'][train_idx], dm['patient_edge_feats'][train_idx],
    dm['block_ids'][train_idx], dm['channel_ids'][train_idx],
    dm['essentiality'][train_idx], dm['essentiality_masks'][train_idx],
)
train_loader = DataLoader(train_ds, batch_size=PRETRAIN_BATCH, shuffle=True, drop_last=True)

val_tensors = tuple(t[val_idx].to(device) for t in [
    dm['node_features'], dm['node_masks'], dm['cancer_types'], dm['clinical'],
    dm['gene_indices'], dm['patient_edge_feats'],
    dm['block_ids'], dm['channel_ids'],
    dm['essentiality'], dm['essentiality_masks'],
])

In [ ]:
# Pre-train
model = AtlasTransformerV6(PRETRAIN_CFG).to(device)
head = EssentialityHead(PRETRAIN_CFG['hidden_dim'], dropout=0.2).to(device)
loss_fn = EssentialityLoss(rank_weight=RANK_WEIGHT)

n_params = sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in head.parameters())
print(f'Parameters: {n_params:,}')

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(head.parameters()),
    lr=PRETRAIN_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRETRAIN_EPOCHS)

best_val_loss = float('inf')
best_corr = 0.0
best_state = None
patience_counter = 0

print(f'\nPre-training on DepMap essentiality ({PRETRAIN_EPOCHS} epochs)...\n')
t0 = time.time()

for epoch in range(PRETRAIN_EPOCHS):
    model.train(); head.train()
    total_loss = 0; n_batches = 0

    for batch in train_loader:
        nf, nm, ct, clin, gi, pef, bi, ci, ess, em = [b.to(device) for b in batch]
        ef = gather_edge_features(dm_gpm, pef, gi, nm)
        nf, nm, ef, ess, em = apply_mutation_dropout(nf, nm, ef, ess, em, MUT_DROP_MIN, MUT_DROP_MAX)

        optimizer.zero_grad()
        node_hidden, _ = model.encode(nf, nm, ct, clin, ef, bi, ci)
        predictions = head(node_hidden)
        loss, metrics = loss_fn(predictions, ess, em)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(head.parameters()), 1.0)
        optimizer.step()
        total_loss += loss.item(); n_batches += 1

    scheduler.step()

    # Validate
    model.eval(); head.eval()
    with torch.no_grad():
        v_nf, v_nm, v_ct, v_clin, v_gi, v_pef, v_bi, v_ci, v_ess, v_em = val_tensors
        v_ef = gather_edge_features(dm_gpm, v_pef, v_gi, v_nm)
        v_hidden, _ = model.encode(v_nf, v_nm, v_ct, v_clin, v_ef, v_bi, v_ci)
        v_pred = head(v_hidden)
        v_loss, v_metrics = loss_fn(v_pred, v_ess, v_em)
        valid = v_em.bool()
        if valid.sum() > 10:
            p = v_pred[valid].cpu().numpy()
            t = v_ess[valid].cpu().numpy()
            corr = float(np.corrcoef(p, t)[0, 1]) if np.std(p) > 0 else 0.0
        else:
            corr = 0.0

    improved = v_loss.item() < best_val_loss
    if improved:
        best_val_loss = v_loss.item()
        best_corr = corr
        best_state = {
            'backbone': {k: v.cpu().clone() for k, v in model.state_dict().items()},
            'config': PRETRAIN_CFG,
            'epoch': epoch + 1,
            'val_loss': best_val_loss,
            'val_corr': best_corr,
        }
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or improved or patience_counter >= PRETRAIN_PATIENCE:
        marker = ' *' if improved else ''
        print(f'  Epoch {epoch+1:3d}: train={total_loss/n_batches:.4f} '
              f'val_mse={v_metrics["mse"]:.4f} corr={corr:.3f} '
              f'best={best_corr:.3f}{marker}')

    if patience_counter >= PRETRAIN_PATIENCE:
        print(f'  Early stop at epoch {epoch+1}')
        break

elapsed = time.time() - t0
print(f'\nPre-training done: {best_state["epoch"]} epochs, corr={best_corr:.3f} [{elapsed:.0f}s]')
torch.save(best_state, 'pretrained_backbone.pt')
print('Saved: pretrained_backbone.pt')

---
## Stage 2: Fine-tune on patient survival (pre-trained vs baseline)

In [ ]:
# Load patient data
pt = torch.load('colab_patient_data.pt', map_location='cpu', weights_only=False)
print(f"Patients: {pt['node_features'].shape[0]}")
print(f"Gene vocab: {len(pt['gene_vocab'])}")
print(f"Cancer types: {pt['n_cancer_types']}")

# Load pre-trained backbone
pretrain_ckpt = torch.load('pretrained_backbone.pt', map_location='cpu', weights_only=False)
print(f"Pre-trained: epoch {pretrain_ckpt['epoch']}, corr={pretrain_ckpt['val_corr']:.3f}")

In [ ]:
# Fine-tuning config
FT_CFG = {
    'hidden_dim': 64,
    'n_heads': 4,
    'n_intra_layers': 2,
    'dropout': 0.1,
    'node_feat_dim': pt['node_features'].shape[-1],  # auto-detect from data
    'edge_feat_dim': pt['graph_edge_dim'] + pt['patient_edge_dim'],
    'n_cancer_types': pt['n_cancer_types'],
    'n_channels': pt['n_channels'],
    'n_blocks': pt['n_blocks'],
}

FT_EPOCHS = 200
FT_LR = 3e-4
FT_BATCH = 256
FT_PATIENCE = 15
PHASE1_EPOCHS = 5
BACKBONE_LR_SCALE = 0.1
N_FOLDS = 5
print(f'Node feat dim: {FT_CFG["node_feat_dim"]}')

In [ ]:
# Move patient data to device
nf = pt['node_features'].to(device)
nm = pt['node_masks'].to(device)
ct = pt['cancer_types'].to(device)
clinical = pt['clinical'].to(device)
atlas_sums = pt['atlas_sums'].to(device)
times = pt['times']
events = pt['events']
pt_gpm = pt['gene_pair_matrix'].to(device)
pt_pef = pt['patient_edge_feats'].to(device)
pt_gi = pt['gene_indices'].to(device)
pt_bi = pt['block_ids'].to(device)
pt_ci = pt['channel_ids'].to(device)

print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.1f}GB' if device.type == 'cuda' else 'CPU')

In [ ]:
SIGN_WEIGHT = 0.5  # weight of sign loss relative to Cox

def train_fold(model, train_idx, val_idx, phase1_epochs=0, backbone_lr_scale=0.1):
    """Train one CV fold with sign x magnitude decomposition.

    Loss = cox_ph_loss(hazard, time, event) + SIGN_WEIGHT * sign_loss(model, nf, nm)
    Cox learns magnitude ordering. Sign loss teaches direction from atlas labels.
    """
    train_t = torch.tensor(train_idx, dtype=torch.long)
    val_t = torch.tensor(val_idx, dtype=torch.long)

    # Phase 1: frozen backbone, train sign+magnitude heads
    if phase1_epochs > 0:
        for name, param in model.named_parameters():
            if any(k in name for k in ['node_encoder', 'intra_block', 'block_to_channel',
                                        'cross_channel', 'channel_pool', 'age_mod']):
                param.requires_grad = False
        trainable = [p for p in model.parameters() if p.requires_grad]
        opt1 = torch.optim.Adam(trainable, lr=FT_LR, weight_decay=1e-4)

        for ep in range(phase1_epochs):
            model.train()
            perm = np.random.permutation(len(train_idx))
            for b_start in range(0, len(perm), FT_BATCH):
                b_abs = train_t[perm[b_start:b_start + FT_BATCH]]
                opt1.zero_grad()
                b_nf = nf[b_abs]
                b_nm = nm[b_abs]
                b_edge = gather_edge_features(pt_gpm, pt_pef[b_abs], pt_gi[b_abs], b_nm)
                hazard = model(b_nf, b_nm, ct[b_abs], clinical[b_abs],
                               atlas_sums[b_abs], b_edge, pt_bi[b_abs], pt_ci[b_abs])
                loss_cox = cox_ph_loss(hazard, times[b_abs].to(device), events[b_abs].to(device))
                loss_sign = sign_loss(model, b_nf, b_nm)
                loss = loss_cox + SIGN_WEIGHT * loss_sign
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                opt1.step()
        for param in model.parameters():
            param.requires_grad = True

    # Phase 2: full training with discriminative LR
    backbone_params, readout_params = [], []
    for name, param in model.named_parameters():
        if any(k in name for k in ['node_encoder', 'intra_block', 'block_to_channel',
                                    'cross_channel', 'channel_pool', 'age_mod']):
            backbone_params.append(param)
        else:
            readout_params.append(param)

    if phase1_epochs > 0:
        param_groups = [
            {'params': backbone_params, 'lr': FT_LR * backbone_lr_scale},
            {'params': readout_params, 'lr': FT_LR},
        ]
    else:
        param_groups = [{'params': list(model.parameters()), 'lr': FT_LR}]

    optimizer = torch.optim.Adam(param_groups, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)

    best_c = 0.0
    no_improve = 0

    for epoch in range(FT_EPOCHS):
        model.train()
        perm = np.random.permutation(len(train_idx))
        epoch_cox = 0; epoch_sign = 0; n_batches = 0

        for b_start in range(0, len(perm), FT_BATCH):
            b_abs = train_t[perm[b_start:b_start + FT_BATCH]]
            optimizer.zero_grad()
            b_nf = nf[b_abs]
            b_nm = nm[b_abs]
            b_edge = gather_edge_features(pt_gpm, pt_pef[b_abs], pt_gi[b_abs], b_nm)
            hazard = model(b_nf, b_nm, ct[b_abs], clinical[b_abs],
                           atlas_sums[b_abs], b_edge, pt_bi[b_abs], pt_ci[b_abs])
            loss_cox = cox_ph_loss(hazard, times[b_abs].to(device), events[b_abs].to(device))
            loss_s = sign_loss(model, b_nf, b_nm)
            loss = loss_cox + SIGN_WEIGHT * loss_s
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_cox += loss_cox.item()
            epoch_sign += loss_s.item()
            n_batches += 1

        scheduler.step()

        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                val_preds = []
                for v_start in range(0, len(val_idx), FT_BATCH):
                    v_abs = val_t[v_start:v_start + FT_BATCH]
                    v_edge = gather_edge_features(pt_gpm, pt_pef[v_abs], pt_gi[v_abs], nm[v_abs])
                    h = model(nf[v_abs], nm[v_abs], ct[v_abs], clinical[v_abs],
                              atlas_sums[v_abs], v_edge, pt_bi[v_abs], pt_ci[v_abs])
                    val_preds.append(h.cpu())
                h_val = torch.cat(val_preds).numpy().flatten()

            e_val = events[val_idx].numpy().astype(bool)
            t_val = times[val_idx].numpy()
            valid = t_val > 0
            try:
                c = concordance_index_censored(e_val[valid], t_val[valid], h_val[valid])[0]
            except Exception:
                c = 0.5

            if c > best_c:
                best_c = c
                no_improve = 0
            else:
                no_improve += 1

            if (epoch + 1) % 20 == 0:
                print(f'      Ep {epoch+1:3d}: cox={epoch_cox/n_batches:.4f} sign={epoch_sign/n_batches:.4f} C={c:.4f} best={best_c:.4f}')

            if no_improve >= FT_PATIENCE // 5:
                print(f'      Early stop ep {epoch+1}, C={best_c:.4f}')
                break

    return best_c

In [ ]:
# Holdback + CV split (same as train_atlas_transformer_v6.py)
n_total = len(events)
np.random.seed(SEED)
all_idx = np.arange(n_total)
np.random.shuffle(all_idx)
n_holdback = int(n_total * 0.15)
holdback_idx = all_idx[:n_holdback]
cv_idx = all_idx[n_holdback:]
events_cv = events[cv_idx].numpy()
print(f'Patients: {n_total}, Holdback: {n_holdback}, CV: {len(cv_idx)}')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
pretrained_results = []
baseline_results = []

In [ ]:
# Run comparison
t0 = time.time()

for fold, (train_rel, val_rel) in enumerate(skf.split(cv_idx, events_cv)):
    train_idx = cv_idx[train_rel]
    val_idx = cv_idx[val_rel]

    # --- Pre-trained ---
    print(f'\n=== Fold {fold} PRE-TRAINED ===')
    model_pt = AtlasTransformerV6(FT_CFG).to(device)
    backbone_state = pretrain_ckpt['backbone']
    model_state = model_pt.state_dict()
    loaded = sum(1 for k, v in backbone_state.items()
                 if k in model_state and model_state[k].shape == v.shape)
    for k, v in backbone_state.items():
        if k in model_state and model_state[k].shape == v.shape:
            model_state[k] = v
    model_pt.load_state_dict(model_state)
    print(f'    Loaded {loaded} params from pre-trained backbone')

    c_pt = train_fold(model_pt, train_idx, val_idx,
                      phase1_epochs=PHASE1_EPOCHS,
                      backbone_lr_scale=BACKBONE_LR_SCALE)
    pretrained_results.append(c_pt)
    print(f'  Fold {fold} pre-trained: C={c_pt:.4f}')

    # --- Baseline ---
    print(f'\n=== Fold {fold} BASELINE ===')
    model_bl = AtlasTransformerV6(FT_CFG).to(device)
    c_bl = train_fold(model_bl, train_idx, val_idx)
    baseline_results.append(c_bl)
    print(f'  Fold {fold} baseline: C={c_bl:.4f}')

    del model_pt, model_bl
    torch.cuda.empty_cache() if device.type == 'cuda' else None

elapsed = time.time() - t0
print(f'\nTotal time: {elapsed/60:.1f} min')

In [ ]:
# Results
pt_mean, pt_std = np.mean(pretrained_results), np.std(pretrained_results)
bl_mean, bl_std = np.mean(baseline_results), np.std(baseline_results)
delta = pt_mean - bl_mean

print('=' * 60)
print(f'  Pre-trained: {pt_mean:.4f} +/- {pt_std:.4f}  {[f"{c:.4f}" for c in pretrained_results]}')
print(f'  Baseline:    {bl_mean:.4f} +/- {bl_std:.4f}  {[f"{c:.4f}" for c in baseline_results]}')
print(f'  Delta:       {delta:+.4f}')
print('=' * 60)

results = {
    'pretrained': {'mean': float(pt_mean), 'std': float(pt_std),
                   'folds': [float(c) for c in pretrained_results]},
    'baseline': {'mean': float(bl_mean), 'std': float(bl_std),
                 'folds': [float(c) for c in baseline_results]},
    'delta': float(delta),
    'pretrain_epoch': pretrain_ckpt['epoch'],
    'pretrain_corr': pretrain_ckpt['val_corr'],
}

with open('finetune_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved: finetune_results.json')

In [ ]:
# Download results
files.download('finetune_results.json')
files.download('pretrained_backbone.pt')